In [1]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    pypdf \
    ragas \
    mlflow \
    ollama

In [2]:
!apt-get update -qq
!apt-get install -y -qq zstd

!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!ollama --version

In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Ollama server started")

Ollama server started


In [5]:
!ollama pull llama3.2:3b

In [6]:
!ollama list

NAME           ID              SIZE      MODIFIED      
llama3.2:3b    a80c4f17acd5    2.0 GB    2 seconds ago    


In [7]:
import os

os.makedirs("documents", exist_ok=True)

documents = {
    "machine_learning.txt": """
Machine learning is a branch of artificial intelligence that enables
computers to learn patterns from data. Supervised learning uses labelled
data, while unsupervised learning identifies patterns without labelled
outputs. Common algorithms include decision trees, support vector machines,
linear regression and neural networks.
""",

    "deep_learning.txt": """
Deep learning is a subset of machine learning based on artificial neural
networks with multiple layers. Deep learning is widely used for image
recognition, natural language processing, speech recognition and
recommendation systems. Popular frameworks include TensorFlow and PyTorch.
""",

    "rag.txt": """
Retrieval-Augmented Generation, or RAG, combines information retrieval
with language generation. A RAG system first retrieves relevant documents
from a knowledge base and then provides the retrieved context to a
language model. This can improve factual grounding and reduce unsupported
answers.
""",

    "mlops.txt": """
MLOps refers to practices for developing, deploying, monitoring and
maintaining machine learning systems. It combines machine learning,
software engineering and operations. Experiment tracking, model
monitoring, version control and automated deployment are common parts
of MLOps.
"""
}

for filename, content in documents.items():
    with open(f"documents/{filename}", "w") as f:
        f.write(content)

print("Documents created:", os.listdir("documents"))

Documents created: ['mlops.txt', 'deep_learning.txt', 'machine_learning.txt', 'rag.txt']


In [8]:
from langchain_community.document_loaders import TextLoader

all_documents = []

for filename in os.listdir("documents"):
    filepath = os.path.join("documents", filename)

    loader = TextLoader(filepath)
    docs = loader.load()

    all_documents.extend(docs)

print("Number of documents:", len(all_documents))

Number of documents: 4


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(all_documents)

print("Number of chunks:", len(chunks))

Number of chunks: 4


In [10]:
!pip install -q sentence-transformers

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")

/tmp/ipykernel_20490/1113191889.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded


In [12]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="w7_multi_document_rag"
)

print("Vector database created")

Vector database created


In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created")

Retriever created


In [14]:
query = "What is Retrieval-Augmented Generation?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)


--- Document 1 ---
Retrieval-Augmented Generation, or RAG, combines information retrieval
with language generation. A RAG system first retrieves relevant documents
from a knowledge base and then provides the retrieved context to a
language model. This can improve factual grounding and reduce unsupported
answers.

--- Document 2 ---
Deep learning is a subset of machine learning based on artificial neural
networks with multiple layers. Deep learning is widely used for image
recognition, natural language processing, speech recognition and
recommendation systems. Popular frameworks include TensorFlow and PyTorch.

--- Document 3 ---
Machine learning is a branch of artificial intelligence that enables
computers to learn patterns from data. Supervised learning uses labelled
data, while unsupervised learning identifies patterns without labelled
outputs. Common algorithms include decision trees, support vector machines,
linear regression and neural networks.


In [15]:
from langchain_community.llms import Ollama

llm = Ollama(
    model="llama3.2:3b"
)

print("Ollama LLM connected")

Ollama LLM connected


/tmp/ipykernel_20490/750808299.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  llm = Ollama(


In [16]:
def rag_query(question):

    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
You are a helpful question-answering assistant.

Answer the question using ONLY the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = llm.invoke(prompt)

    return answer, retrieved_docs

In [17]:
question = "What is RAG and how does it work?"

answer, sources = rag_query(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

print("\nSources:")
for doc in sources:
    print(doc.metadata.get("source"))

Question:
What is RAG and how does it work?

Answer:
RAG (Retrieval-Augmented Generation) is a system that combines information retrieval with language generation. It works by first retrieving relevant documents from a knowledge base and then providing the retrieved context to a language model. This can improve factual grounding and reduce unsupported answers.

Sources:
documents/rag.txt
documents/mlops.txt
documents/machine_learning.txt


In [18]:
questions = [
    "What is machine learning?",
    "What is deep learning?",
    "What is Retrieval-Augmented Generation?",
    "What is MLOps?",
    "Which technologies are commonly used for deep learning?"
]

for question in questions:

    answer, sources = rag_query(question)

    print("\n" + "=" * 60)
    print("QUESTION:", question)
    print("ANSWER:", answer)


QUESTION: What is machine learning?
ANSWER: Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data.

QUESTION: What is deep learning?
ANSWER: Deep learning is a subset of machine learning based on artificial neural networks with multiple layers, widely used for image recognition, natural language processing, speech recognition and recommendation systems.

QUESTION: What is Retrieval-Augmented Generation?
ANSWER: Retrieval-Augmented Generation (RAG) is a system that combines information retrieval with language generation. It first retrieves relevant documents from a knowledge base and then provides the retrieved context to a language model.

QUESTION: What is MLOps?
ANSWER: MLOps refers to practices for developing, deploying, monitoring and maintaining machine learning systems, combining machine learning, software engineering, and operations.

QUESTION: Which technologies are commonly used for deep learning?
ANSWER: TensorFlow and PyT

In [19]:
import mlflow

mlflow.set_experiment("W7_Multi_Document_RAG")

with mlflow.start_run():

    mlflow.log_param("llm", "llama3.2:3b")
    mlflow.log_param("embedding_model", "all-MiniLM-L6-v2")
    mlflow.log_param("retrieval_k", 3)
    mlflow.log_param("number_of_documents", len(all_documents))
    mlflow.log_param("number_of_chunks", len(chunks))

    test_question = "What is MLOps?"

    answer, sources = rag_query(test_question)

    mlflow.log_param(
        "test_question",
        test_question
    )

    mlflow.log_metric(
        "retrieved_documents",
        len(sources)
    )

    print("MLflow run completed")

MLflow run completed


In [20]:
evaluation_questions = [
    "What is machine learning?",
    "What is deep learning?",
    "What is RAG?",
    "What is MLOps?"
]

results = []

for question in evaluation_questions:

    answer, sources = rag_query(question)

    results.append({
        "question": question,
        "answer": answer,
        "retrieved_documents": len(sources)
    })

results

[{'question': 'What is machine learning?',
  'answer': 'Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data.',
  'retrieved_documents': 3},
 {'question': 'What is deep learning?',
  'answer': 'Deep learning is a subset of machine learning based on artificial neural networks with multiple layers.',
  'retrieved_documents': 3},
 {'question': 'What is RAG?',
  'answer': 'RAG stands for Retrieval-Augmented Generation.',
  'retrieved_documents': 3},
 {'question': 'What is MLOps?',
  'answer': 'MLOps refers to practices for developing, deploying, monitoring and maintaining machine learning systems.',
  'retrieved_documents': 3}]

In [22]:
!pip uninstall -y ragas langchain langchain-community langchain-core langchain-openai -q

!pip install -q \
    "ragas==0.2.15" \
    "langchain==0.3.25" \
    "langchain-community==0.3.24" \
    "langchain-core==0.3.59"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.59 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.8 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.59 which is incompatible.
langgraph-sdk 0.4.4 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.59 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.59 which is incompatible.


In [23]:
import ragas
import langchain
import langchain_community

print("Ragas:", ragas.__version__)
print("LangChain:", langchain.__version__)
print("LangChain Community:", langchain_community.__version__)

Ragas: 0.2.15
LangChain: 0.3.25
LangChain Community: 0.3.24


In [24]:
from ragas import EvaluationDataset

dataset = EvaluationDataset.from_list([
    {
        "user_input": r["question"],
        "retrieved_contexts": [
            doc.page_content
            for doc in retriever.invoke(r["question"])
        ],
        "response": r["answer"]
    }
    for r in results
])

print(dataset)

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response'], len=4)


In [25]:
assert len(all_documents) >= 4
assert len(chunks) > 0
assert retriever is not None
assert llm is not None

print("All basic tests passed successfully!")

All basic tests passed successfully!


In [26]:
question = "How does RAG improve the reliability of language model answers?"

answer, sources = rag_query(question)

print("FINAL QUESTION")
print(question)

print("\nFINAL ANSWER")
print(answer)

print("\nRETRIEVED SOURCES")
for source in sources:
    print("-", source.metadata.get("source"))

FINAL QUESTION
How does RAG improve the reliability of language model answers?

FINAL ANSWER
According to the context, RAG improves the reliability of language model answers by providing the retrieved context from relevant documents to the language model, which can improve factual grounding and reduce unsupported answers.

RETRIEVED SOURCES
- documents/rag.txt
- documents/deep_learning.txt
- documents/mlops.txt
